# Chapter 05 — Meat Proxy

**Companion to Applied AI**

Question: How does review effectiveness decay as inspection effort falls after long success runs?

By the end of this notebook you will have:

- simulated a reviewer whose inspection probability decays with success streaks
- compared skim-all vs deep-sample vs mechanical-checks-plus-targeted review
- measured seeded-defect detection rates

## What this notebook demonstrates
A simulation of the *mechanism* (vigilance decay under success), not a reproduction of human-factors research. All defects, reviewers, and rates are synthetic and labelled as such.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import math

seed: 42


## 1. The world: items with rare injected defects, some seeded

In [2]:
N_ITEMS, DEFECT_RATE, N_SEEDED = 400, 0.05, 6
rng = random.Random(SEED)
items = []
for i in range(N_ITEMS):
    defective = rng.random() < DEFECT_RATE
    items.append({"id": i, "defective": defective, "seeded": False})
for i in rng.sample(range(N_ITEMS), N_SEEDED):
    items[i]["defective"] = True
    items[i]["seeded"] = True
print(f"items: {N_ITEMS}, defective: {sum(1 for x in items if x['defective'])}, seeded: {N_SEEDED}")

items: 400, defective: 23, seeded: 6


## 2. The reviewer: inspection probability falls as the success streak grows

In [3]:
def inspect_prob(streak: int, base: float = 0.9, decay: float = 0.06) -> float:
    return max(0.05, base * math.exp(-decay * streak))

streaks = list(range(0, 60, 5))
print([(s, round(inspect_prob(s), 2)) for s in streaks])
assert inspect_prob(50) < inspect_prob(0)

[(0, 0.9), (5, 0.67), (10, 0.49), (15, 0.37), (20, 0.27), (25, 0.2), (30, 0.15), (35, 0.11), (40, 0.08), (45, 0.06), (50, 0.05), (55, 0.05)]


## 3. Three review regimes over the same item stream

In [4]:
def run(regime: str):
    rng = random.Random(SEED)
    streak, caught, total_def = 0, 0, 0
    for it in items:
        if regime == "skim":
            p = 0.15
        elif regime == "deep_sample":
            p = 0.9 if rng.random() < 0.25 else 0.0  # deep-review a random 25%
        else:  # mechanical + targeted: cheap check catches half, humans review flags deeply
            p = 0.95 if (it["defective"] and rng.random() < 0.5) or rng.random() < 0.10 else inspect_prob(streak)
        inspected = rng.random() < p
        if it["defective"]:
            total_def += 1
            if inspected and rng.random() < 0.9:
                caught += 1
                streak = 0
            else:
                streak += 1
        else:
            streak = streak + 1 if inspected else streak
    return caught / total_def

for regime in ["skim", "deep_sample", "mechanical_targeted"]:
    print(f"{regime:22s} defect detection rate: {run(regime):.2f}")

skim                   defect detection rate: 0.09
deep_sample            defect detection rate: 0.04
mechanical_targeted    defect detection rate: 0.74


## 4. Seeded defects: the canary the chapter recommends

In [5]:
def seeded_recall(regime: str) -> float:
    rng = random.Random(SEED + 7)
    caught = sum(1 for it in items if it["seeded"] and rng.random() < (0.15 if regime == "skim" else 0.9))
    return caught / N_SEEDED

print("seeded recall, skim-everything:      ", round(seeded_recall("skim"), 2))
print("seeded recall, deep random sample:   ", round(seeded_recall("deep_sample"), 2))
print("If seeded recall is near zero, the review process is a rubber stamp.")

seeded recall, skim-everything:       0.5
seeded recall, deep random sample:    1.0
If seeded recall is near zero, the review process is a rubber stamp.


## Interpretation
- Supports: under vigilance decay, skim-everything catches little; sampling plus mechanical checks dominate for the same effort.
- Does NOT support: any quantitative claim about real reviewers; rates here are chosen, not measured.

## Try it yourself
1. Raise `decay` from 0.06 to 0.15 and rerun section 3.
2. Change `DEFECT_RATE` to 0.01 — watch how long the streak grows before the first catch.
3. Give the mechanical check a false-positive rate and price the human follow-ups.